In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import warnings
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. PORTFOLIO DEFINITIONS & PATHS AUDIT CHECKING
# ==============================================================================
# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"

PORTFOLIO = {
    'binance-futures': {
        'symbol': 'BTCUSDT',
        'bbo': DATA_DIR / r"binf_BTC_bbo.csv",
        'tardis_dir': DATA_DIR / "binance-futures" / "BTCUSDT" / "derivative_ticker" / "year=2025"
    },
    'bybit': {
        'symbol': 'BTCUSDT',
        'bbo': DATA_DIR / r"byde_BTC_bbo.csv",
        'tardis_dir': DATA_DIR / "bybit" / "BTCUSDT" / "derivative_ticker" / "year=2025"
    },
    'okex-swap': {
        'symbol': 'BTC-USDT-SWAP',
        'bbo': DATA_DIR / r"okex_BTC_bbo.csv",
        'tardis_dir': DATA_DIR / "okex-swap" / "BTC-USDT-SWAP" / "derivative_ticker" / "year=2025"
    },
    'gate-io-futures': {
        'symbol': 'BTC_USDT',
        'bbo': DATA_DIR / r"gate_BTC_bbo.csv",
        'tardis_dir': DATA_DIR / "gate-io-futures" / "BTC_USDT" / "derivative_ticker" / "year=2025"
    },
    'hyperliquid': {
        'symbol': 'BTC',
        'bbo': DATA_DIR / r"hype_BTC_bbo.csv",
        'tardis_dir': DATA_DIR / "hyperliquid" / "BTC" / "derivative_ticker" / "year=2025"
    },
    'kucoin-futures': {
        'symbol': 'BTCUSDTM',
        'bbo': DATA_DIR / r"kusw_BTC_bbo.csv",
        'tardis_dir': DATA_DIR / "kucoin-futures" / "BTCUSDTM" / "derivative_ticker" / "year=2025"
    }
}

# ==============================================================================
# 2. TIME PARSING UTILITIES
# ==============================================================================
def robust_to_datetime(series):
    s_num = pd.to_numeric(series, errors='coerce')
    med_val = s_num.median()
    if pd.isna(med_val): 
        return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_localize(None)
    if med_val > 1e16:   unit = 'ns'
    elif med_val > 1e13: unit = 'us'
    elif med_val > 1e10: unit = 'ms'
    else:                unit = 's'
    return pd.to_datetime(s_num, unit=unit, errors='coerce', utc=True).dt.tz_localize(None)

# ==============================================================================
# 3. GLOBAL PORTFOLIO AUDIT ENGINE
# ==============================================================================
def run_global_portfolio_audit():
    print("=====================================================================================")
    print(" 🕵️ PORTFOLIO-WIDE HIGH-FREQUENCY DATA AUDIT: AUROS BBO vs TARDIS DERIVATIVES")
    print("=====================================================================================\n")
    
    bbo_summary = []
    tardis_summary = []
    
    for exch, paths in PORTFOLIO.items():
        print(f"🔄 Auditing Raw Datasets for: {exch.upper()}...")
        
        # ---------------------------------------------------------
        # STAGE A: AUROS BBO CSV FILE AUDIT
        # ---------------------------------------------------------
        bbo_path = paths['bbo']
        if bbo_path.exists():
            try:
                # Optimized chunked/low-memory timestamp extract
                b_df = pd.read_csv(bbo_path, usecols=['minute', 'bid', 'ask'])
                b_time = robust_to_datetime(b_df['minute']).sort_values()
                
                b_start = b_time.min()
                b_end = b_time.max()
                b_gaps = b_time.diff().dt.total_seconds() / 60.0
                b_max_gap = b_gaps.max()
                
                # Check for empty orderbooks
                b_df['mid'] = (b_df['bid'] + b_df['ask']) / 2
                bbo_nulls = b_df['mid'].isna().mean() * 100
                
                bbo_summary.append({
                    'Exchange': exch.upper(),
                    'Start': b_start,
                    'End': b_end,
                    'Max Gap (Mins)': b_max_gap,
                    'Null Mid%': f"{bbo_nulls:.2f}%"
                })
            except Exception as e:
                print(f"  [!] Error reading BBO CSV for {exch}: {str(e)}")
        else:
            print(f"  [!] BBO CSV File Not Found: {bbo_path.name}")

        # ---------------------------------------------------------
        # STAGE B: TARDIS DERIVATIVE TICKER DIRECTORY SCAN
        # ---------------------------------------------------------
        tardis_dir = paths['tardis_dir']
        if tardis_dir.exists():
            files = list(tardis_dir.glob("*.parquet"))
            if files:
                t_times = []
                
                # Structural tracking vectors for column health
                col_counts = {'last_price': 0, 'index_price': 0, 'mark_price': 0, 'funding_rate': 0}
                col_nulls = {'last_price': 0, 'index_price': 0, 'mark_price': 0, 'funding_rate': 0}
                total_tardis_rows = 0
                
                for f in tqdm(files, desc=f"  Scanning Tardis Parquet Files", leave=False):
                    try:
                        # Inspect columns present inside the file layout without reading data
                        parquet_file = pq.ParquetFile(f)
                        available_cols = parquet_file.schema.names
                        
                        cols_to_load = ['timestamp']
                        for c in col_counts.keys():
                            if c in available_cols:
                                cols_to_load.append(c)
                                
                        f_df = pd.read_parquet(f, columns=cols_to_load)
                        total_tardis_rows += len(f_df)
                        
                        # Process timestamps for coverage mapping
                        t_times.append(robust_to_datetime(f_df['timestamp']))
                        
                        # Process null densities for every target matrix variable
                        for c in col_counts.keys():
                            if c in f_df.columns:
                                col_counts[c] += len(f_df)
                                col_nulls[c] += f_df[c].isna().sum()
                            else:
                                # Treat completely absent columns as 100% missing for this file block
                                col_nulls[c] += len(f_df)
                                
                    except Exception:
                        continue
                
                if t_times:
                    t_series = pd.concat(t_times).sort_values()
                    t_start = t_series.min()
                    t_end = t_series.max()
                    t_gaps = t_series.diff().dt.total_seconds() / 60.0
                    t_max_gap = t_gaps.max()
                    
                    # Convert accumulated null profiles into final metrics
                    null_metrics = {}
                    for c in col_counts.keys():
                        pct = (col_nulls[c] / total_tardis_rows * 100) if total_tardis_rows > 0 else 100.0
                        null_metrics[c] = f"{pct:.2f}%"
                        
                    tardis_summary.append({
                        'Exchange': exch.upper(),
                        'Start': t_start,
                        'End': t_end,
                        'Max Gap(m)': t_max_gap,
                        'Null Last%': null_metrics['last_price'],
                        'Null Index%': null_metrics['index_price'],
                        'Null Mark%': null_metrics['mark_price'],
                        'Null Fund%': null_metrics['funding_rate']
                    })
            else:
                print(f"  [!] No parquet files found in: {tardis_dir}")
        else:
            print(f"  [!] Tardis Directory Not Found: {tardis_dir}")
            
    # ==============================================================================
    # 4. PRINT CONSOLIDATED PORTFOLIO SUMMARY REPORTS
    # ==============================================================================
    print("\n" + "="*95)
    print(" 📊 SUMMARY REPORT A: INSTITUTIONAL BBO DATA (AUROS MULTI-VENUE)")
    print("="*95)
    if bbo_summary:
        print(pd.DataFrame(bbo_summary).to_string(index=False))
    else:
        print("No active BBO profiles analyzed successfully.")
    print("="*95 + "\n")

    print("="*115)
    print(" 📊 SUMMARY REPORT B: DERIVATIVE TICKER VARIABLE DATA DENSITY (TARDIS PORTFOLIO)")
    print("="*115)
    if tardis_summary:
        print(pd.DataFrame(tardis_summary).to_string(index=False))
    else:
        print("No active Tardis portfolios scanned successfully.")
    print("="*115 + "\n")
    
    print("💡 Next Step: Inspect the printed matrices to pinpoint the exact intersection windows")
    print("   and confirm which adjusted price columns are safe to build inside your thesis matrix.")

if __name__ == "__main__":
    run_global_portfolio_audit()

 🕵️ PORTFOLIO-WIDE HIGH-FREQUENCY DATA AUDIT: AUROS BBO vs TARDIS DERIVATIVES

🔄 Auditing Raw Datasets for: BINANCE-FUTURES...


  Scanning Tardis Parquet Files:   0%|          | 0/306 [00:00<?, ?it/s]

🔄 Auditing Raw Datasets for: BYBIT...


  Scanning Tardis Parquet Files:   0%|          | 0/306 [00:00<?, ?it/s]

🔄 Auditing Raw Datasets for: OKEX-SWAP...


...

🔄 Auditing Raw Datasets for: GATE-IO-FUTURES...


  Scanning Tardis Parquet Files:   0%|          | 0/306 [00:00<?, ?it/s]

🔄 Auditing Raw Datasets for: HYPERLIQUID...


  Scanning Tardis Parquet Files:   0%|          | 0/306 [00:00<?, ?it/s]

🔄 Auditing Raw Datasets for: KUCOIN-FUTURES...
  [!] Tardis Directory Not Found: C:\Users\tz_ho\research_data\kucoin-futures\BTCUSDTM\derivative_ticker\year=2025

 📊 SUMMARY REPORT A: INSTITUTIONAL BBO DATA (AUROS MULTI-VENUE)
       Exchange               Start                 End  Max Gap (Mins) Null Mid%
BINANCE-FUTURES 2025-03-01 00:00:00 2026-02-28 23:59:00            69.0     0.00%
          BYBIT 2025-03-01 00:00:00 2025-07-16 11:35:00             1.0     0.00%
      OKEX-SWAP 2025-03-01 00:00:00 2026-02-28 23:59:00          1767.0     0.00%
GATE-IO-FUTURES 2025-03-01 00:00:00 2026-02-28 23:59:00            66.0     0.00%
    HYPERLIQUID 2025-04-30 07:51:00 2026-02-25 22:16:00          8102.0     0.00%
 KUCOIN-FUTURES 2025-03-01 00:00:00 2026-02-28 23:59:00            66.0     0.00%

 📊 SUMMARY REPORT B: DERIVATIVE TICKER VARIABLE DATA DENSITY (TARDIS PORTFOLIO)
       Exchange                      Start                        End  Max Gap(m) Null Last% Null Index% Null Mark% Nu

In [7]:
#V1

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import itertools
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURATION & PATHS
# ==============================================================================
# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"

OUTPUT_PARQUET = DATA_DIR / "BTC_TAGGED_MASTER.parquet"

TARDIS_SYMBOLS = {
    'binance-futures': 'BTCUSDT', 'bybit': 'BTCUSDT', 'gate-io-futures': 'BTC_USDT',
    'hyperliquid': 'BTC', 'kucoin-futures': 'BTCUSDTM', 'okex-swap': 'BTC-USDT-SWAP'
}

BBO_PATHS = {
    'binance-futures': DATA_DIR / r"binf_BTC_bbo.csv",
    'bybit': DATA_DIR / r"byde_BTC_bbo.csv",
    'okex-swap': DATA_DIR / r"okex_BTC_bbo.csv",
    'gate-io-futures': DATA_DIR / r"gate_BTC_bbo.csv",
    'hyperliquid': DATA_DIR / r"hype_BTC_bbo.csv",
    'kucoin-futures': DATA_DIR / r"kusw_BTC_bbo.csv"
}

FUNDING_HISTORY_FILE = DATA_DIR / "funding_rate_history_202604231452.csv"

# The Full Annual Scope
GLOBAL_START = pd.Timestamp('2025-03-01 00:00:00')
GLOBAL_END = pd.Timestamp('2026-02-2 23:59:00')
STALENESS_THRESHOLD_MINS = 3

def robust_to_datetime(series):
    s_num = pd.to_numeric(series, errors='coerce')
    med_val = s_num.median()
    if pd.isna(med_val): dt_series = pd.to_datetime(series, errors='coerce', utc=True)
    else:
        if med_val > 1e16:   unit = 'ns'
        elif med_val > 1e13: unit = 'us'
        elif med_val > 1e10: unit = 'ms'
        else:                unit = 's'
        dt_series = pd.to_datetime(s_num, unit=unit, errors='coerce', utc=True)
    
    # [FIX] Removed .dt.floor('1min') to prevent premature time-shifting and lookahead bias
    return dt_series.dt.tz_localize(None)

# ==============================================================================
# 2. INGESTION MODULES
# ==============================================================================
def compile_tardis(exch, symbol):
    # Target the parent directory instead of a specific year
    tardis_base_dir = DATA_DIR / exch / symbol / "derivative_ticker"
    if not tardis_base_dir.exists(): return None
    dfs = []
    
    # Use .rglob() to recursively find all parquets across all year folders
    for file in tardis_base_dir.rglob("*.parquet"):
        try:
            df = pd.read_parquet(file)
            df['timestamp'] = robust_to_datetime(df['timestamp'])
            df.set_index('timestamp', inplace=True)
            cols = ['funding_rate', 'index_price', 'mark_price']
            
            dfs.append(df[[c for c in cols if c in df.columns]].resample('1min', label='right').last())
        except: continue
        
    if not dfs: return None
    annual_df = pd.concat(dfs)
    annual_df = annual_df[~annual_df.index.duplicated(keep='last')].sort_index()
    if 'index_price' not in annual_df.columns or annual_df['index_price'].isna().all():
        annual_df['index_price'] = annual_df.get('mark_price', np.nan)
    else: 
        annual_df['index_price'] = annual_df['index_price'].fillna(annual_df.get('mark_price', np.nan))
    return annual_df


def compile_bbo(path):
    if not path.exists(): return None
    df = pd.read_csv(path)
    df['timestamp'] = robust_to_datetime(df['minute'])
    df.set_index('timestamp', inplace=True)
    
    # [FIX] Added label='right' to align with reality
    df = df.resample('1min', label='right').last()
    
    df.loc[df['bid'] >= df['ask'], ['bid', 'ask']] = np.nan
    df['mid'] = (df['bid'] + df['ask']) / 2
    return df[['bid', 'ask', 'mid']]

def compile_funding(path, exch_code):
    df = pd.read_csv(path)
    mask = (df['exchange'] == exch_code) & (df['symbol'].str.contains('BTC', na=False))
    f_df = df[mask].copy()
    if f_df.empty: return pd.DataFrame()
    time_col = next((c for c in f_df.columns if 'time' in c.lower()), None)
    f_df['datetime'] = robust_to_datetime(f_df[time_col])
    f_df = f_df.dropna(subset=['datetime']).sort_values('datetime').set_index('datetime')
    f_df = f_df[~f_df.index.duplicated(keep='last')]
    rate_col = next((c for c in f_df.columns if 'rate' in c.lower() and 'time' not in c.lower()), None)
    f_df['interval_mins'] = f_df.index.to_series().diff().dt.total_seconds() / 60.0
    f_df['interval_mins'] = f_df['interval_mins'].fillna(f_df['interval_mins'].mode()[0])
    f_df['epoch_id'] = range(1, len(f_df) + 1)
    return f_df[[rate_col, 'interval_mins', 'epoch_id']].rename(columns={rate_col: 'hist_rate'})

# ==============================================================================
# 3. TAGGED PIPELINE BUILDER
# ==============================================================================
def build_tagged_pipeline():
    print("🚀 Compiling Full Tagged Master Matrix...")
    processed_exchanges = []
    exch_map = {'binance-futures': 'binf', 'bybit': 'byde', 'okex-swap': 'okex', 'hyperliquid': 'hype', 'gate-io-futures':'gate', 'kucoin-futures':'kusw'}
    
    # Pre-generate the master timeline grid
    master_idx = pd.date_range(start=GLOBAL_START, end=GLOBAL_END, freq='1min')
    
    for exch, tardis_sym in TARDIS_SYMBOLS.items():
        print(f"  [{exch.upper()}] Processing...")
        t_df = compile_tardis(exch, tardis_sym)
        b_df = compile_bbo(BBO_PATHS.get(exch))
        f_df = compile_funding(FUNDING_HISTORY_FILE, exch_map.get(exch, exch))
        
        if t_df is None or b_df is None: continue
            
        # 1. Join & Reindex to full timeline
        exch_df = t_df.join(b_df, how='outer').reindex(master_idx)
        
        # 2. CREATE THE STALENESS TAGS (Before forward filling)
        # We define data as "stale" if the mid price hasn't updated, OR if it's currently NaN
        is_flatlining = (exch_df['mid'].diff() == 0)
        is_missing = exch_df['mid'].isna()
        
        # Count consecutive flatlines. The cumsum trick resets whenever it's NOT flatlining.
        consecutive_stale_mins = is_flatlining.groupby((~is_flatlining).cumsum()).cumsum()
        
        # The Tag: True if missing, OR if flatlining longer than threshold
        exch_df[f'stale_{exch}'] = is_missing | (consecutive_stale_mins > STALENESS_THRESHOLD_MINS)
        
        # 3. Massive Forward Fill (Now safe because it's tagged)
        # Forward fill the price columns infinitely
        fill_cols = ['bid', 'ask', 'mid', 'index_price', 'mark_price']
        exch_df[fill_cols] = exch_df[fill_cols].ffill()
        
        # 4. Funding Accrual Math
        exch_df = exch_df.join(f_df, how='left')
        exch_df['epoch_id'] = exch_df['epoch_id'].ffill()
        exch_df['interval_mins'] = exch_df['interval_mins'].ffill()
        settlement_times = pd.Series(exch_df.index, index=exch_df.index).where(~exch_df['hist_rate'].isna()).ffill()
        exch_df['fraction_elapsed'] = np.clip(((exch_df.index - settlement_times).dt.total_seconds() / 60.0) / exch_df['interval_mins'], 0.0, 1.0)
        
        exch_df['fr_v1_ffill'] = exch_df['funding_rate'].fillna(exch_df['hist_rate'] if 'hist_rate' in exch_df.columns else np.nan).ffill()
        
        if 'mark_price' in exch_df.columns and 'index_price' in exch_df.columns and 'mid' in exch_df.columns:
            exch_df['inst_mark_prem'] = (exch_df['mark_price'] - exch_df['index_price']) / exch_df['index_price']
            exch_df['inst_mid_prem'] = (exch_df['mid'] - exch_df['index_price']) / exch_df['index_price']
            exch_df['fr_v2_mark_avg'] = exch_df.groupby('epoch_id')['inst_mark_prem'].expanding().mean().reset_index(level=0, drop=True)
            exch_df['fr_v3_mid_avg'] = exch_df.groupby('epoch_id')['inst_mid_prem'].expanding().mean().reset_index(level=0, drop=True)
        else:
            exch_df['fr_v2_mark_avg'], exch_df['fr_v3_mid_avg'] = exch_df['fr_v1_ffill'], exch_df['fr_v1_ffill']

        # 5. Price Adjustments (Price + Accrual)
        bases = {'idx': 'index_price', 'mark': 'mark_price'}
        rates = {'ffill': 'fr_v1_ffill', 'mark_avg': 'fr_v2_mark_avg', 'mid_avg': 'fr_v3_mid_avg'}
        for b_n, b_c in bases.items():
            for r_n, r_c in rates.items():
                if b_c not in exch_df.columns: continue
                accrual = exch_df[b_c] * exch_df[r_c] * exch_df['fraction_elapsed']
                for t_n, t_c in {'mid':'mid'}.items(): # Only adjust mid to save memory/space
                    if t_c not in exch_df.columns: continue
                    exch_df[f'padj_{t_n}_{b_n}_{r_n}_{exch}'] = exch_df[t_c] + accrual

        # ==============================================================================
        # 🟢 THE FIX: Rename and retain the epoch_id column
        # ==============================================================================
        if 'epoch_id' in exch_df.columns and 'fr_v1_ffill' in exch_df.columns:
            exch_df.rename(columns={'epoch_id': f'epoch_id_{exch}', 'fr_v1_ffill': f'fr_v1_{exch}'}, inplace=True)
            cols = [c for c in exch_df.columns if c.startswith('padj_')] + ['mid', f'stale_{exch}', f'epoch_id_{exch}', f'fr_v1_{exch}']
        else:
            cols = [c for c in exch_df.columns if c.startswith('padj_')] + ['mid', f'stale_{exch}']
            
        processed_exchanges.append(exch_df[cols].rename(columns={'mid': f'mid_{exch}'}))
    
    # 6. Global Merge
    print("🔗 Stacking Aligned Venue Matrices...")
    final_df = pd.concat(processed_exchanges, axis=1)
    
    # 7. Spread Computation
    print(f"📈 Computing Spreads...")
    exchs = [c.split('_')[-1] for c in final_df.columns if c.startswith('mid_')]
    for exch_A, exch_B in itertools.combinations(exchs, 2):
        pair = f"{exch_A}_vs_{exch_B}"
        
        # Raw Spread
        if f'mid_{exch_A}' in final_df.columns and f'mid_{exch_B}' in final_df.columns:
            final_df[f'spread_raw_mid_{pair}'] = (np.log(final_df[f'mid_{exch_A}']) - np.log(final_df[f'mid_{exch_B}'])) * 10000
            
        # Adjusted Spread
        for b, r in itertools.product(bases.keys(), rates.keys()):
            cA, cB = f'padj_mid_{b}_{r}_{exch_A}', f'padj_mid_{b}_{r}_{exch_B}'
            if cA in final_df.columns and cB in final_df.columns:
                final_df[f'spread_padj_{b}_{r}_{pair}'] = (np.log(final_df[cA]) - np.log(final_df[cB])) * 10000
                    
    final_df.to_parquet(OUTPUT_PARQUET)
    print(f"\n💾 Saved Tagged Master Matrix: {OUTPUT_PARQUET.name} | Shape: {final_df.shape}")

if __name__ == "__main__":
    build_tagged_pipeline()

🚀 Compiling Full Tagged Master Matrix...
  [BINANCE-FUTURES] Processing...
  [BYBIT] Processing...
  [GATE-IO-FUTURES] Processing...
  [HYPERLIQUID] Processing...
  [KUCOIN-FUTURES] Processing...
  [OKEX-SWAP] Processing...
🔗 Stacking Aligned Venue Matrices...
📈 Computing Spreads...

💾 Saved Tagged Master Matrix: BTC_TAGGED_MASTER.parquet | Shape: (488160, 120)
